# Detecção Hierárquica de Toxicidade no Civil Comments
## CNN + Bi-LSTM com seleção *nested* de thresholds

**Notebook final executável da iniciação científica**

Este é o artefato canônico para reproduzir, auditar e publicar o experimento final. Execute de cima para baixo em um runtime limpo, preferencialmente com GPU, e salve a versão final **com os outputs preservados**. A implementação permanece em `src/` e `scripts/`; o notebook registra proveniência, executa o runner oficial e transforma o output observado em tabelas e gráficos.

**Resultado documentado de referência:** Macro F1 end-to-end `0.4412`; repetição independente `0.4427`. A execução deste notebook é reportada separadamente para evitar cherry-picking.

## 1. Desenho experimental

```text
Civil Comments → Stage 1 (toxicity/severe_toxicity) → gate → Stage 2 multilabel
                                                       ├ obscene
                                                       ├ threat
                                                       ├ insult
                                                       ├ identity_attack
                                                       └ sexual_explicit
```

A verdade de referência roteia quando `toxicity >= 0.4`. Os thresholds de **predição** são escolhidos somente na validação interna; os folds externos são usados apenas para avaliação. O runner final usa 2 folds externos e até 5 épocas por estágio com EarlyStopping.

## 2. Preparar um clone limpo

O notebook usa uma pasta isolada e sempre sincroniza `main`. Isso evita branch antiga, clone aninhado e estado oculto do notebook.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
REPO_URL='https://github.com/Umbura/Hatespeech_Detection_Civil_Comments_NLP_Obsolete.git'
BASE=Path('/content') if Path('/content').exists() else Path.cwd()
REPO=BASE/'hatespeech-final-reproduction'
if REPO.exists():
    if not (REPO/'.git').exists(): raise RuntimeError(f'{REPO} não é um clone Git.')
    dirty=subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()
    if dirty: raise RuntimeError('O clone de reprodução possui alterações locais.')
    subprocess.run(['git','fetch','origin','--prune'],cwd=REPO,check=True)
    subprocess.run(['git','switch','main'],cwd=REPO,check=True)
    subprocess.run(['git','pull','--ff-only','origin','main'],cwd=REPO,check=True)
else:
    subprocess.run(['git','clone','--branch','main','--single-branch',REPO_URL,str(REPO)],check=True)
os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('Repository:',REPO); print('Commit:',COMMIT)

## 3. Ambiente registrado

Instala as versões usadas nas execuções Colab documentadas. Se o ambiente solicitar reinício após `pip`, reinicie e execute novamente desde a primeira célula.

In [ ]:
PACKAGES=['tensorflow==2.20.0','numpy==2.0.2','pandas==2.2.3','scikit-learn==1.6.1','imbalanced-learn==0.14.2','datasets==5.0.0','iterative-stratification==0.1.9']
subprocess.run([sys.executable,'-m','pip','install','-q',*PACKAGES],check=True)
subprocess.run([sys.executable,'-m','pip','check'],check=True)

## 4. Proveniência

Registra software, hardware, commit e um SHA-256 agregado dos arquivos científicos. O hash de conteúdo continua útil mesmo se o histórico Git for reorganizado depois.

In [ ]:
import hashlib, importlib.metadata as meta, json, platform
from datetime import datetime, timezone
import numpy as np, pandas as pd, sklearn, tensorflow as tf
FILES=['scripts/run_hierarchical_cv.py','src/hate_speech_detection/cv_pipeline.py','src/hate_speech_detection/hierarchical_splits.py','src/hate_speech_detection/target_strategy.py','src/hate_speech_detection/threshold_selection.py','requirements.txt']
def sha(p):
    h=hashlib.sha256(); h.update(Path(p).read_bytes()); return h.hexdigest()
hashes={x:sha(REPO/x) for x in FILES}
manifest=hashlib.sha256('\n'.join(f'{x}\0{hashes[x]}' for x in sorted(hashes)).encode()).hexdigest()
gpus=tf.config.list_physical_devices('GPU')
hardware='CPU'
if gpus and shutil.which('nvidia-smi'):
    q=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
    hardware=q.stdout.strip() or str(gpus)
runtime={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'git_commit':COMMIT,'scientific_manifest_sha256':manifest,'python':platform.python_version(),'tensorflow':tf.__version__,'numpy':np.__version__,'pandas':pd.__version__,'scikit_learn':sklearn.__version__,'datasets':meta.version('datasets'),'hardware':hardware}
display(pd.DataFrame(runtime.items(),columns=['Item','Valor']))

## 5. Validação estrutural antes do treino

In [ ]:
for cmd in [[sys.executable,'-m','pip','check'],[sys.executable,'-m','compileall','-q','src','scripts'],[sys.executable,'-m','unittest','discover','-s','tests','-p','test_*.py','-v']]:
    print('$',' '.join(cmd)); subprocess.run(cmd,cwd=REPO,check=True)
print('\nRepository validation: PASSED')

## 6. Benchmark documentado

Os valores de referência são lidos de `results/final_metrics.json`; não são duplicados manualmente no código do notebook.

In [ ]:
ref=json.loads((REPO/'results'/'final_metrics.json').read_text(encoding='utf-8'))
primary,rep=ref['primary_run'],ref['replication_run']
reference=pd.DataFrame([('Stage 1 F1 nested',primary['stage1_nested']['f1']),('Stage 1 recall nested',primary['stage1_nested']['recall']),('Stage 1 PR-AUC/AP',primary['stage1_nested']['average_precision']),('Stage 1 ROC-AUC',primary['stage1_nested']['roc_auc']),('Stage 2 oracle Macro F1',primary['stage2_oracle_nested_macro_f1']),('End-to-end Macro F1 fixed',primary['end_to_end_fixed_macro_f1']),('End-to-end Macro F1 nested',primary['end_to_end_nested_macro_f1']),('Repetição end-to-end Macro F1',rep['end_to_end_nested_macro_f1'])],columns=['Métrica','Referência'])
display(reference.style.format({'Referência':'{:.4f}'}))

## 7. Execução completa

Executa `python scripts/run_hierarchical_cv.py --n-splits 2 --epochs 5` no dataset completo. O log integral é salvo fora do clone Git. Em CPU funciona, mas GPU é recomendada.

In [ ]:
import re,time
RUN=BASE/'hate_speech_final_run'; RUN.mkdir(exist_ok=True)
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); log_path=RUN/f'full_{stamp}.log'
cmd=[sys.executable,'scripts/run_hierarchical_cv.py','--n-splits','2','--epochs','5']
show=('Fold ','Stage 1 rows:','Stage 2 rows:','best inner-validation epoch:','Selected inner thresholds:','--- Ground-truth','--- Threshold selection','--- Stage 1','--- Stage 2','--- End-to-end','Macro F1:','Accuracy:','Precision:','Recall:','Routing rate:','PR-AUC','ROC-AUC:','obscene F1:','threat F1:','insult F1:','identity_attack F1:','sexual_explicit F1:','Routed samples:','any Stage 2 label:')
started=time.perf_counter(); lines=[]
p=subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env={**os.environ,'PYTHONUNBUFFERED':'1'})
assert p.stdout is not None
with log_path.open('w',encoding='utf-8') as f:
    for line in p.stdout:
        lines.append(line); f.write(line); f.flush()
        if any(x in line for x in show): print(line.rstrip(),flush=True)
rc=p.wait(); elapsed=time.perf_counter()-started
if rc: raise RuntimeError(f'Full run failed ({rc}). Veja {log_path}')
full_log=''.join(lines)
print(f'\nFull experiment: PASSED — {elapsed/60:.1f} min')

## 8. Extrair os resultados observados

In [ ]:
def sec(title):
    m=re.search(rf'--- {re.escape(title)} ---\s*(.*?)(?=\n--- |\Z)',full_log,re.S)
    if not m: raise ValueError(f'Seção ausente: {title}')
    return m.group(1)
def stage1(title):
    d={}
    for line in sec(title).splitlines():
        if ': ' in line:
            k,v=line.split(': ',1)
            try:d[k]=float(v.split(' | ')[0])
            except ValueError:pass
    return d
LABELS=['obscene','threat','insult','identity_attack','sexual_explicit']
def stage2(title):
    text=sec(title); macro=float(re.search(r'^Macro F1:\s*([0-9.]+)',text,re.M).group(1)); per={}
    for label in LABELS:
        m=re.search(rf'^{label} F1:\s*([0-9.]+)(?:\s*\|\s*PR-AUC/AP:\s*([0-9.]+|n/a))?',text,re.M)
        if not m: raise ValueError(f'{label} ausente em {title}')
        per[label]={'f1':float(m.group(1))}
        if m.group(2) and m.group(2)!='n/a': per[label]['average_precision']=float(m.group(2))
    return {'macro_f1':macro,'per_label':per}
s1_fixed=stage1('Stage 1 toxicity gate — fixed prediction threshold 0.40')
s1_nested=stage1('Stage 1 toxicity gate — nested tuned routing threshold')
s2_fixed=stage2('Stage 2 oracle — fixed label threshold 0.50')
s2_nested=stage2('Stage 2 oracle — nested tuned label thresholds')
e2e_fixed=stage2('End-to-end — fixed 0.40 routing / 0.50 labels')
e2e_nested=stage2('End-to-end — nested tuned thresholds')

## 9. Stage 1

In [ ]:
s1_table=pd.DataFrame({'Fixed 0.40':{'Accuracy':s1_fixed['Accuracy'],'Precision':s1_fixed['Precision'],'Recall':s1_fixed['Recall'],'F1':s1_fixed['F1'],'Routing rate':s1_fixed['Routing rate'],'PR-AUC/AP':s1_fixed['PR-AUC (average precision)'],'ROC-AUC':s1_fixed['ROC-AUC'],'severe_toxicity MAE':s1_fixed['severe_toxicity auxiliary MAE']},'Nested':{'Accuracy':s1_nested['Accuracy'],'Precision':s1_nested['Precision'],'Recall':s1_nested['Recall'],'F1':s1_nested['F1'],'Routing rate':s1_nested['Routing rate'],'PR-AUC/AP':s1_nested['PR-AUC (average precision)'],'ROC-AUC':s1_nested['ROC-AUC'],'severe_toxicity MAE':s1_nested['severe_toxicity auxiliary MAE']}})
display(s1_table.style.format('{:.4f}'))

## 10. Stage 2 oracle e sistema end-to-end

`Stage 2 oracle` mede o classificador multilabel assumindo roteamento correto. O **end-to-end** é a métrica do sistema completo.

In [ ]:
s2_table=pd.DataFrame({'Fixed':[s2_fixed['per_label'][x]['f1'] for x in LABELS],'Nested':[s2_nested['per_label'][x]['f1'] for x in LABELS]},index=LABELS); s2_table.loc['Macro F1']=[s2_fixed['macro_f1'],s2_nested['macro_f1']]
e2e_table=pd.DataFrame({'Fixed':[e2e_fixed['per_label'][x]['f1'] for x in LABELS],'Nested':[e2e_nested['per_label'][x]['f1'] for x in LABELS]},index=LABELS); e2e_table.loc['Macro F1']=[e2e_fixed['macro_f1'],e2e_nested['macro_f1']]
ap_table=pd.DataFrame({'Fixed AP':[s2_fixed['per_label'][x].get('average_precision',float('nan')) for x in LABELS],'Nested AP':[s2_nested['per_label'][x].get('average_precision',float('nan')) for x in LABELS]},index=LABELS)
print('Stage 2 oracle — F1'); display(s2_table.style.format('{:.4f}'))
print('Stage 2 oracle — PR-AUC/AP'); display(ap_table.style.format('{:.4f}'))
print('End-to-end'); display(e2e_table.style.format('{:.4f}'))
gain=e2e_nested['macro_f1']-e2e_fixed['macro_f1']; print(f'Ganho absoluto: {gain:+.4f} | ganho relativo: {gain/e2e_fixed["macro_f1"]*100:+.1f}%')

## 11. Thresholds por fold e cobertura do gate

In [ ]:
threshold_text=sec('Threshold selection summary')
fold_re=re.compile(r'Fold (\d+): routing=([0-9.]+), inner end-to-end Macro F1=([0-9.]+), inner gate recall=([0-9.]+), inner routing rate=([0-9.]+)')
label_re=re.compile(r'^\s+(obscene|threat|insult|identity_attack|sexual_explicit): threshold=([0-9.]+), inner oracle F1=([0-9.]+)',re.M)
fold_matches=list(fold_re.finditer(threshold_text)); folds=[]; label_threshold_rows=[]
for i,m in enumerate(fold_matches):
    folds.append({'Fold':int(m.group(1)),'Routing threshold':float(m.group(2)),'Inner E2E Macro F1':float(m.group(3)),'Inner gate recall':float(m.group(4)),'Inner routing rate':float(m.group(5))})
    start=m.end(); end=fold_matches[i+1].start() if i+1<len(fold_matches) else len(threshold_text)
    for label,thr,f1 in label_re.findall(threshold_text[start:end]): label_threshold_rows.append({'Fold':int(m.group(1)),'Label':label,'Threshold':float(thr),'Inner oracle F1':float(f1)})
print('Routing threshold por fold'); display(pd.DataFrame(folds).style.format(precision=4))
print('Stage 2 thresholds por fold'); display(pd.DataFrame(label_threshold_rows).style.format({'Threshold':'{:.2f}','Inner oracle F1':'{:.4f}'}))
coverage=sec('Ground-truth gate coverage analysis')
print(coverage)

## 12. Visualizações geradas da execução

In [ ]:
import matplotlib.pyplot as plt
summary=pd.DataFrame({'Fixed':[s1_fixed['F1'],s2_fixed['macro_f1'],e2e_fixed['macro_f1']],'Nested':[s1_nested['F1'],s2_nested['macro_f1'],e2e_nested['macro_f1']]},index=['Stage 1 F1','Stage 2 Oracle Macro F1','End-to-End Macro F1'])
ax=summary.plot(kind='bar',figsize=(9,5)); ax.set_ylim(0,0.8); ax.set_ylabel('F1'); ax.set_title('Threshold fixo vs. seleção nested'); ax.grid(axis='y',alpha=.25); plt.xticks(rotation=15,ha='right'); plt.tight_layout(); plt.show()
ax=e2e_table.drop(index='Macro F1').plot(kind='bar',figsize=(10,5)); ax.set_ylim(0,.85); ax.set_ylabel('F1'); ax.set_title('End-to-end F1 por label'); ax.grid(axis='y',alpha=.25); plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.show()

## 13. Comparação com os resultados documentados

In [ ]:
official=float(primary['end_to_end_nested_macro_f1']); repeated=float(rep['end_to_end_nested_macro_f1']); current=float(e2e_nested['macro_f1'])
comparison=pd.DataFrame([('Benchmark principal',official,0),('Repetição documentada',repeated,repeated-official),('Execução deste notebook',current,current-official)],columns=['Execução','End-to-end Macro F1','Delta vs. principal'])
display(comparison.style.format({'End-to-end Macro F1':'{:.4f}','Delta vs. principal':'{:+.4f}'}))

## 14. Limitações e conclusão

O projeto usa dois folds externos por custo computacional; ainda há propagação de erro do Stage 1 para o Stage 2; `sexual_explicit` é a categoria mais afetada pelo gate de verdade; fairness, robustez e desempenho em um teste oficial congelado não foram estabelecidos. Não há alegação de estado da arte ou prontidão para produção.

A contribuição central é a avaliação hierárquica metodologicamente reparada e o ganho obtido ao selecionar thresholds somente na validação interna, sem trocar a arquitetura CNN + Bi-LSTM.

## 15. Checklist de publicação

Depois de `Run all`: confirme `Repository validation: PASSED` e `Full experiment: PASSED`, verifique as tabelas/gráficos, mantenha os outputs, salve o notebook executado e envie essa versão para `notebooks/final/HateSpeech_Final_Hierarchical.ipynb`.

In [ ]:
status=subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()
print('Git working tree:', 'clean' if not status else 'modified')
print(f'End-to-end Macro F1 observado: {e2e_nested["macro_f1"]:.4f}')
print('Scientific manifest SHA-256:',manifest)